# HeartMuLa — free song-generation server for BibleMusically

Runs the **open-source, Apache-2.0 [HeartMuLa](https://github.com/HeartMuLa/heartlib)** model (a 2026 Suno competitor, commercially usable / YouTube-safe) on a free Kaggle/Colab GPU and exposes the **same task API as the ACE-Step server** (`/release_task` → `/query_result` → `/v1/audio`), so the app drives it with the existing engine.

Paste the printed URL into **Settings → Music engine → HeartMuLa → server URL**, then set the engine to *HeartMuLa*.

**Before you run:** enable GPU — Kaggle: *Settings → Accelerator → GPU T4 x2*; Colab: *Runtime → GPU*.

> **First-run notes.** HeartMuLa has no official server, so this notebook wraps its `HeartMuLaGenPipeline` in a small FastAPI server. The pipeline import path / signature may differ slightly by release — if startup errors on the import, adjust the import line in the last cell (the README lists the documented API). Generating a full song on a single T4 can take a few minutes; keep song length modest so it finishes inside the app's poll window.


## 1. Install HeartMuLa (heartlib) + server deps


In [ ]:
# ── Upstream revision: report, and pin if the app asked for one ──────────────
# Rewritten by the app at push time from this engine's "Upstream revision" setting. Empty means
# track whatever upstream has published, which is the default and what every run did before.
_UPSTREAM_PINS = {}

def _upstream(path, name):
    """Check out a pinned ref if there is one, then print the revision actually in use."""
    import subprocess as _sp
    _ref = _UPSTREAM_PINS.get(name)
    if _ref:
        try:
            _sp.run(['git', '-C', path, 'fetch', '--depth', '1', 'origin', _ref], check=True)
            _sp.run(['git', '-C', path, 'checkout', '--detach', 'FETCH_HEAD'], check=True)
            print(f'[upstream] {name} pinned to {_ref}', flush=True)
        except Exception as _ex:
            print(f'[upstream] {name} could not be pinned to {_ref} ({_ex}) - using default branch', flush=True)
    try:
        _sha = _sp.run(['git', '-C', path, 'rev-parse', '--short', 'HEAD'],
                       capture_output=True, text=True).stdout.strip()
        _date = _sp.run(['git', '-C', path, 'log', '-1', '--format=%cs'],
                        capture_output=True, text=True).stdout.strip()
        print(f'[upstream] {name} {_sha} {_date}', flush=True)
    except Exception as _ex:
        print(f'[upstream] {name} revision unknown ({_ex})', flush=True)

import os, sys, subprocess, shutil

def sh(*args):
    subprocess.run(list(args), check=True)

# Clone into heartlib_src, NOT ./heartlib: the repo keeps its code in src/heartlib/,
# so a ./heartlib dir in the notebook cwd shadows the pip-installed package as an
# empty namespace package and every `from heartlib import ...` silently breaks.
if os.path.isdir('heartlib') and not os.path.isdir('heartlib_src'):
    shutil.move('heartlib', 'heartlib_src')   # migrate an old clone out of the way
if not os.path.isdir('heartlib_src'):
    sh('git','clone','--depth','1','https://github.com/HeartMuLa/heartlib.git','heartlib_src')
_upstream('heartlib_src', 'heartlib')

# heartlib's pyproject.toml HARD-PINS its deps (incl. transformers==4.57.0 and
# traitlets==5.7.1). pip honors those exact pins, so on Colab/Kaggle you WILL see:
#   * a "yanked version: transformers 4.57.0" WARNING  -> required by heartlib; it installs & works.
#   * "dependency conflicts" vs tpot/google-colab/moviepy/gym/pandas -> pre-installed base
#     packages we never import. Changing them would BREAK heartlib, not fix anything.
# Both messages are non-fatal noise. The verification block below is the real success signal.
# NON-editable install: `-e` registers its import hook via a .pth file that only
# loads at interpreter startup, so the same kernel process that ran the install
# could never `import heartlib` (the cause of the 'No module named heartlib' error).
sh(sys.executable,'-m','pip','install','-q','./heartlib_src')
sh(sys.executable,'-m','pip','install','-q','fastapi','uvicorn','huggingface_hub')

# ---- Verify the environment actually works (this, not pip's red text, is what matters) ----
import importlib
importlib.invalidate_caches()  # pick up the just-installed package in this same process
ok = True
for mod in ('torch','transformers','tokenizers'):
    try:
        m = importlib.import_module(mod)
        print(f'  ✅ {mod:14s} {getattr(m, "__version__", "ok")}')
    except Exception as ex:
        ok = False
        print(f'  ❌ {mod:14s} FAILED: {ex}')
# Import the CLASS, not just the module — an empty namespace package would pass a
# bare `import heartlib` and hide the problem (exactly what bit the first run).
try:
    from heartlib import HeartMuLaGenPipeline
    import heartlib
    print(f'  ✅ heartlib       HeartMuLaGenPipeline importable ({heartlib.__file__})')
except Exception as ex:
    ok = False
    print(f'  ❌ heartlib       FAILED: {ex!r}')
    print('     (if this mentions a namespace/shadow issue: remove any ./heartlib dir and re-run)')
print('\nheartlib installed — environment OK. Proceed to the checkpoint download cell.'
      if ok else '\n⚠️  A check FAILED above — fix that error before continuing.')


## 2. Download checkpoints (Apache-2.0)


In [ ]:
import os, shutil, subprocess, sys

# ── Disk-space-aware checkpoint download ────────────────────────────
# The three checkpoints total ~25-30 GB. On Kaggle, /kaggle/working is capped at
# ~19.5 GB → "No space left on device". The root disk (/tmp) has ~50+ GB free, so
# download there and symlink ./ckpt to it — the serve cell's './ckpt' keeps working.

def free_gb(path):
    try:
        return shutil.disk_usage(path).free / 2**30
    except OSError:
        return 0.0

candidates = ['/kaggle/tmp', '/tmp', os.getcwd()]
base = max(candidates, key=free_gb)
print('free disk:', {c: f'{free_gb(c):.1f} GB' for c in candidates})

if base == os.getcwd():
    CKPT = os.path.join(os.getcwd(), 'ckpt')
else:
    CKPT = os.path.join(base, 'heartmula_ckpt')
    link = os.path.join(os.getcwd(), 'ckpt')
    # Replace any previous ./ckpt (e.g. a partial download that filled the disk).
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link):
        print('removing previous partial ./ckpt to reclaim space...')
        shutil.rmtree(link)
    os.makedirs(CKPT, exist_ok=True)
    os.symlink(CKPT, link)
    print(f'./ckpt -> {CKPT}')

if free_gb(CKPT) < 35:
    print(f'⚠️  Only {free_gb(CKPT):.1f} GB free — the ~30 GB of checkpoints may not fit. '
          'Restart the session (to clear disk) and re-run if the download fails.')

# check=True so a failed download STOPS here instead of pretending it worked.
downloads = [
    ('HeartMuLa/HeartMuLaGen',                    CKPT),
    ('HeartMuLa/HeartMuLa-oss-3B-happy-new-year', os.path.join(CKPT, 'HeartMuLa-oss-3B')),
    ('HeartMuLa/HeartCodec-oss-20260123',         os.path.join(CKPT, 'HeartCodec-oss')),
]
for repo, dest in downloads:
    print(f'↓ {repo} -> {dest}')
    subprocess.run(['hf', 'download', '--local-dir', dest, repo], check=True)
    # Drop the .cache staging dir hf leaves inside local-dir downloads (frees GBs).
    cache = os.path.join(dest, '.cache')
    if os.path.isdir(cache):
        shutil.rmtree(cache, ignore_errors=True)

total = sum(os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(CKPT) for f in fs) / 2**30
print(f'✅ checkpoints downloaded to ./ckpt ({total:.1f} GB, {free_gb(CKPT):.1f} GB still free)')


## 3. Install cloudflared (public tunnel)


In [ ]:
import subprocess
if subprocess.run(['which','cloudflared'], capture_output=True).returncode != 0:
    subprocess.run('curl -L --output /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /tmp/cloudflared.deb', shell=True, check=True)
print('cloudflared ready.')

## 4. (Optional) API key

Leave blank for the simplest setup; if set, paste the same value into the app's API key field.


In [ ]:
API_KEY = ''  # must match the app's HeartMuLa API key field if set
print('key set' if API_KEY else 'open server')

## 5. Warm-load the model + launch the server + tunnel

Loads `HeartMuLaGenPipeline` once, then serves the ACE-Step-compatible task API. Keep this cell running.


In [ ]:
import os, io, json, time, uuid, threading, subprocess, re
import torch

# ---- Warm-load the pipeline once (adjust the import if your heartlib release differs) ----
pipe = None
load_error = None
try:
    try:
        from heartlib import HeartMuLaGenPipeline
    except Exception as first_ex:
        try:
            from heartlib.pipelines.music_generation import HeartMuLaGenPipeline  # real location in src/heartlib
        except Exception:
            raise first_ex  # surface the ORIGINAL error, not the fallback's
    # heartlib types `device` as Union[torch.device, Dict[str, torch.device]] and later runs
    # `torch.autocast(device_type=self.mula_device.type, ...)`. A plain 'cuda' STRING loads
    # fine (nn.Module.to() accepts one), so you still get 'HeartMuLa pipeline loaded.' — but
    # every generation then fails with: 'str' object has no attribute 'type'.
    # It must be a real torch.device.
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipe = HeartMuLaGenPipeline.from_pretrained(
        './ckpt',
        device={'mula': dev, 'codec': dev},
        dtype={'mula': torch.bfloat16, 'codec': torch.float16},
        version='3B',
        lazy_load=True,
    )
    print('HeartMuLa pipeline loaded.')
except Exception as ex:
    load_error = str(ex)
    print('WARNING: could not load pipeline:', load_error)
    print('Fix the import/from_pretrained call in this cell per the heartlib README, then re-run.')

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse, FileResponse
import uvicorn

PORT = 8003
AUDIO_DIR = '/kaggle/working/heartmula_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
app = FastAPI()
TASKS = {}  # task_id -> {status:0/1/2, file:str, duration:float}

def _auth(request):
    if API_KEY and request.headers.get('authorization','') != f'Bearer {API_KEY}':
        raise HTTPException(status_code=401, detail='bad api key')

def _run(task_id, tags, lyrics, dur_ms, topk=50, temperature=1.0, cfg_scale=1.5):
    try:
        base = os.path.join(AUDIO_DIR, task_id)
        tags_p, lyr_p, out_p = base+'.tags.txt', base+'.lyrics.txt', base+'.mp3'
        open(tags_p,'w').write(tags)
        open(lyr_p,'w').write(lyrics or '[Verse]\n')
        with torch.no_grad():
            # Sampler settings come from the app now. They were fixed here, which meant the
            # quality controls in Settings did not reach the model at all.
            pipe({'lyrics': lyr_p, 'tags': tags_p}, max_audio_length_ms=int(dur_ms),
                 save_path=out_p, topk=int(topk), temperature=float(temperature),
                 cfg_scale=float(cfg_scale))
        # What the model actually produced, which is not always what was asked for: a 500s
        # request has come back as a 120s track. ffprobe ships in the Kaggle image; if it is
        # somehow missing, fall back to the request rather than failing a good render.
        _made = dur_ms / 1000.0
        try:
            _p = subprocess.run(
                ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                 '-of', 'default=nw=1:nk=1', out_p],
                capture_output=True, text=True, timeout=60)
            if _p.returncode == 0 and _p.stdout.strip():
                _made = float(_p.stdout.strip())
        except Exception as _ex:
            print('could not measure the rendered audio, reporting the requested length:', _ex)
        if abs(_made - dur_ms / 1000.0) > 5:
            print(f'note: asked for {dur_ms/1000.0:.0f}s, produced {_made:.1f}s', flush=True)
        TASKS[task_id] = {'status': 1, 'file': f'/v1/audio?path={task_id}.mp3', 'duration': _made}
    except Exception as ex:
        import traceback; traceback.print_exc()
        print('generation failed:', ex)
        # Store the error so the app can show WHY straight from the job log (not just the notebook).
        TASKS[task_id] = {'status': 2, 'file': '', 'duration': 0.0, 'error': f'{type(ex).__name__}: {ex}'}

@app.post('/release_task')
async def release_task(request: Request):
    _auth(request)
    if pipe is None:
        raise HTTPException(status_code=503, detail=f'model not loaded: {load_error}')
    body = await request.json()
    tags = (body.get('prompt') or body.get('caption') or '').strip()
    lyrics = (body.get('lyrics') or '').strip()
    dur = float(body.get('audio_duration') or body.get('duration') or 180.0)
    task_id = uuid.uuid4().hex
    TASKS[task_id] = {'status': 0, 'file': '', 'duration': dur}
    topk = body.get('topk', 50)
    temperature = body.get('temperature', 1.0)
    cfg_scale = body.get('cfg_scale', 1.5)
    threading.Thread(target=_run,
                     args=(task_id, tags, lyrics, dur*1000.0, topk, temperature, cfg_scale),
                     daemon=True).start()
    return JSONResponse({'data': {'task_id': task_id, 'status': 'queued'}, 'code': 200})

@app.post('/query_result')
async def query_result(request: Request):
    _auth(request)
    body = await request.json()
    out = []
    for tid in body.get('task_id_list', []):
        t = TASKS.get(tid)
        if not t:
            out.append({'task_id': tid, 'status': 2, 'result': json.dumps({'status': 2})})
        else:
            out.append({'task_id': tid, 'status': t['status'],
                        'result': json.dumps({'file': t['file'], 'status': t['status'], 'duration': t['duration'], 'error': t.get('error','')})})
    return JSONResponse({'data': out, 'code': 200})

@app.get('/v1/audio')
def get_audio(path: str):
    if not re.fullmatch(r'[0-9a-f]+\.mp3', path):
        raise HTTPException(status_code=400, detail='bad path')
    fp = os.path.join(AUDIO_DIR, path)
    if not os.path.isfile(fp):
        raise HTTPException(status_code=404, detail='not ready')
    return FileResponse(fp, media_type='audio/mpeg')

def _serve():
    uvicorn.run(app, host='127.0.0.1', port=PORT, log_level='warning')
threading.Thread(target=_serve, daemon=True).start()

import urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:{PORT}/query_result',
            data=json.dumps({'task_id_list':[]}).encode(), headers={'Content-Type':'application/json'}), timeout=3)
        print('server up.'); break
    except urllib.error.HTTPError:
        print('server up.'); break
    except Exception:
        time.sleep(2)

# ── Batch-run guard v2 ──────────────────────────────────────────────
# Source-update pushes run GPU-less and should exit fast; a GPU batch run is a
# DELIBERATE server start (the app's "Start server" button pushes with GPU on)
# and must open the tunnel and keep serving.
_is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive').lower() == 'batch'

# Two independent questions, asked separately because they fail apart — and which one failed IS
# the diagnosis:
#   nvidia-smi  — does this container have a GPU and a working driver at all?
#   torch.cuda  — can the framework that runs the model actually reach it?
# A GPU-off source-update push answers no to both, and so does Kaggle declining an accelerator.
# An install step that replaced Kaggle's CUDA torch with a CPU-only wheel answers YES to the
# first and no to the second. The old guard asked only nvidia-smi and reported every "no" as an
# exhausted weekly quota — which sent people to a quota page that had 29.8 of 30 hours left on it.
_smi_rc, _smi_note = None, ''
try:
    _smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=120)
    _smi_rc = _smi.returncode
    _smi_note = (_smi.stderr or '').strip().replace('\n', ' ')[:200]
except FileNotFoundError:
    _smi_note = 'nvidia-smi is not installed on this container'
except Exception as _ex:
    _smi_note = '{}: {}'.format(type(_ex).__name__, _ex)
try:
    import torch as _t
    _torch_cuda, _torch_ver = bool(_t.cuda.is_available()), _t.__version__
except Exception as _ex:
    _torch_cuda, _torch_ver = False, 'unavailable ({})'.format(type(_ex).__name__)

# Serving is gated on torch rather than on the driver, because the model is loaded onto whatever
# device torch reports. A container that has a GPU torch cannot see would otherwise open a public
# tunnel to a server generating on CPU — minutes of audio at hours of wall clock, which is a worse
# outcome than not starting, and much harder to diagnose from the app.
_has_gpu = _torch_cuda
if _is_batch and not _has_gpu:
    print('=' * 70)
    print('  NO GPU ON THIS RUN — not serving.')
    print('  nvidia-smi: ' + ('exit {}'.format(_smi_rc) if _smi_rc is not None else 'did not run')
          + (' — {}'.format(_smi_note) if _smi_note else ''))
    print('  torch {}: cuda.is_available() = {}'.format(_torch_ver, _torch_cuda))
    print('  CUDA_VISIBLE_DEVICES = {!r}'.format(os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')))
    if _smi_rc == 0:
        print('  GPU PRESENT BUT TORCH CANNOT USE IT — an install step in this notebook replaced')
        print("  Kaggle's CUDA build of torch with a CPU-only one. Fix that cell; the quota is")
        print('  not the problem here.')
    else:
        print('  KAGGLE GAVE THIS SESSION NO ACCELERATOR. The app always asks for one, so this is')
        print('  the scheduler declining: the weekly quota is spent, both GPU session slots are')
        print('  busy, or no GPU was free at that moment. That last case is common and transient —')
        print('  if the quota page still shows hours left, simply start again.')
        print('  Quota: https://www.kaggle.com/settings  (Accelerator usage).')
    print('  (A deliberate GPU-off push is just a cheap source update - nothing is wrong.)')
    print('=' * 70)
    print('Start the server from the app (Start server button) or run interactively with GPU on.')
else:
    import urllib.request, urllib.error

    # ── Reliable public tunnel with self-healing ───────────────────────────────
    # The failure this fixes: cloudflared prints a *.trycloudflare.com URL and even registers an
    # edge connection, yet the Cloudflare edge never actually ROUTES the hostname, so the URL never
    # answers and the app times out. Quick tunnels are flaky per-process and QUIC (UDP) egress can
    # be throttled. So we: (1) probe our OWN public URL to confirm it truly routes, (2) auto-restart
    # cloudflared — first over QUIC, then over HTTP/2, which survives UDP throttling — and (3) fall
    # back to localhost.run (ssh) if cloudflared keeps failing. Only a URL that actually ANSWERS is
    # printed as ready, and every step is logged so a failure is diagnosable from the app's log tail.
    _url_re = re.compile(r'https://[-a-z0-9]+\.(?:trycloudflare\.com|lhr\.life|serveo\.net)')

    def _probe_public(url, timeout=8):
        # True iff the tunnel truly routes: ANY HTTP status < 500 back proves the edge reached our
        # server. A connection error/timeout, or Cloudflare's own 5xx (e.g. 530 = tunnel down),
        # means "not routed yet".
        try:
            with urllib.request.urlopen(url.rstrip('/') + '/', timeout=timeout) as r:
                return r.status < 500
        except urllib.error.HTTPError as he:
            return he.code < 500
        except Exception:
            return False

    def _pump(proc, holder, tag='tunnel'):
        def _run():
            for line in proc.stdout:
                print(f'[{tag}] {line}', end='')
                m = _url_re.search(line)
                if m and not holder.get('url'):
                    holder['url'] = m.group(0)
        threading.Thread(target=_run, daemon=True).start()

    def _spawn_cloudflared(protocol):
        args = ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}']
        if protocol:
            args += ['--protocol', protocol]
        print(f'[tunnel] launching cloudflared (protocol={protocol or "auto"})...', flush=True)
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _spawn_localhostrun():
        print('[tunnel] launching localhost.run over ssh...', flush=True)
        p = subprocess.Popen(
            ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'UserKnownHostsFile=/dev/null',
             '-o', 'ServerAliveInterval=30', '-R', f'80:localhost:{PORT}', 'nokey@localhost.run'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _bring_up(proc, holder, url_wait=30, route_wait=75):
        t0 = time.time()
        while time.time() - t0 < url_wait and not holder.get('url'):
            if proc.poll() is not None:
                print('[tunnel] process exited before printing a URL.', flush=True); return None
            time.sleep(1)
        url = holder.get('url')
        if not url:
            print(f'[tunnel] no URL within {url_wait}s.', flush=True); return None
        print(f'Tunnel URL: {url}', flush=True)
        print('Waiting for the edge to route it...', flush=True)
        t1 = time.time()
        while time.time() - t1 < route_wait:
            if proc.poll() is not None:
                print('[tunnel] tunnel process exited during the routing wait.', flush=True); return None
            if _probe_public(url):
                print(f'[tunnel] OK: answered from inside Kaggle after {int(time.time()-t1)}s (a brand-new address can still take minutes to route elsewhere).', flush=True)
                return url
            time.sleep(4)
        print(f'[tunnel] {url} never answered within {route_wait}s - treating as dead.', flush=True)
        return None

    _attempts = [('cf', 'quic'), ('cf', 'http2'), ('lhr', None)]
    active_proc = None; public_url = None
    for _i, (_kind, _proto) in enumerate(_attempts, 1):
        print(f'\n[tunnel] ===== attempt {_i}/{len(_attempts)}: {_kind} {_proto or ""} =====', flush=True)
        try:
            _p, _h = _spawn_cloudflared(_proto) if _kind == 'cf' else _spawn_localhostrun()
        except FileNotFoundError as _e:
            print(f'[tunnel] cannot launch ({_e}); skipping this attempt.', flush=True); continue
        _routed = _bring_up(_p, _h)
        if _routed:
            active_proc, public_url = _p, _routed; break
        try: _p.terminate()
        except Exception: pass
        time.sleep(2)

    print('\n' + '=' * 70)
    if public_url:
        print('  PASTE THIS INTO THE APP  ->  Settings -> HeartMuLa server URL:')
        print(f'  {public_url}')
    else:
        print('  FAILED: no working public tunnel after all attempts. The local server is fine,')
        print('  but nothing outside can reach it - retry "Start & connect" from the app.')
    print('=' * 70, flush=True)

    if public_url and active_proc:
        # ── Idle-shutdown watchdog ──────────────────────────────────────
        # After IDLE_SHUTDOWN_MIN minutes with no ESTABLISHED connection to the server port, stop the
        # tunnel so a forgotten run stops burning GPU quota. App polling / liveness counts as activity.
        IDLE_SHUTDOWN_MIN = 15
        def _idle_watchdog():
            _port_hex = ':%04X' % PORT
            _last = time.time()
            while True:
                time.sleep(30)
                _active = False
                for _tbl in ('/proc/net/tcp', '/proc/net/tcp6'):
                    try:
                        with open(_tbl) as _f:
                            for _l in _f.readlines()[1:]:
                                _q = _l.split()
                                if _q[1].endswith(_port_hex) and _q[3] == '01':
                                    _active = True; break
                    except OSError:
                        _active = True
                    if _active: break
                if _active:
                    _last = time.time()
                elif time.time() - _last > IDLE_SHUTDOWN_MIN * 60:
                    print(f'[watchdog] No requests for {IDLE_SHUTDOWN_MIN} min - shutting down to save GPU quota.', flush=True)
                    try: active_proc.terminate()
                    except Exception: pass
                    return
        threading.Thread(target=_idle_watchdog, daemon=True).start()
        print(f'Keep this cell running. Idle watchdog armed: auto-stops after {IDLE_SHUTDOWN_MIN} min idle.', flush=True)
        active_proc.wait()